# 01 — Referencia de distribución XADC

Preparado en 1.7 para ejecutarse en **1.8 sobre la PYNQ-Z2**. No ejecutado todavía
en placa. Valida instalación, procedencia, manifiesto y pareja BIT/HWH, y permite
una carga genérica explícita. No configura el driver XADC heredado, no captura
micrófonos y no lee registros PDM. La aceptación global de fase 1 corresponde a 1.9.

Ejecutar en orden desde un kernel reiniciado y desde una carpeta de notebooks
instalados, fuera del checkout SW. Guardar este notebook con sus resultados en la
carpeta de evidencias creada abajo. No habilitar programación antes de comprobar
que se desea reemplazar el overlay que pueda estar cargado.

## 1. Parámetros e identidad del kernel

Completar `EXPECTED_SW_COMMIT` con el SHA SW publicado en el handoff vigente.
Se deja vacío porque un notebook versionado no puede contener el hash de su propio
commit. El release y hash HW sí están fijados. El entorno observado del servicio
Jupyter es `/usr/local/share/pynq-venv`; comprobarlo dentro del kernel.

In [ ]:
import datetime
import hashlib
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import re
import shlex
import subprocess
import sys
import sysconfig
import tempfile
import uuid

EXPECTED_SW_COMMIT = ""  # copiar SHA completo del handoff vigente
EXPECTED_ENV = Path("/usr/local/share/pynq-venv")
EXPECTED_HW_TAG = "v0.1.0-rc.2"
EXPECTED_MANIFEST_SHA256 = "0d372c66a798cedcc4420caf800c612d34c7720a40f1480620c9809d9d421571"
PROGRAM_FPGA = False  # cambiar explícitamente solo para la carga de 1.8
INSTALL_PACKAGE = False  # requiere volver a False y reiniciar el kernel después
EVIDENCE_ROOT = Path.cwd() / "evidence"
run_id = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
run_dir = EVIDENCE_ROOT / ("01_baseline-" + run_id)
run_dir.mkdir(parents=True, exist_ok=False)
report = {
    "schema_version": 1, "milestone": "phase-1-baseline", "run_id": run_id,
    "started_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version, "executable": sys.executable, "prefix": sys.prefix,
    "uid": os.geteuid(), "declared_board": os.environ.get("BOARD"),
    "site_packages_writable": os.access(sysconfig.get_paths()["purelib"], os.W_OK),
    "fpga_firmware_writable": os.access("/sys/class/fpga_manager/fpga0/firmware", os.W_OK),
    "checks": {}, "overlay_programmed": False,
    "repeated_branch_reinstall_on_board": "not_run_requires_external_install_receipts",
    "phase_acceptance": "not_accepted_requires_1.9_review",
    "pdm_runtime_identity": "not_applicable_legacy_xadc", "build_id_128": None,
}

def save_report():
    (run_dir / "acceptance-report.json").write_text(json.dumps(report, indent=2, default=str) + "\n")

def check(name, operation):
    try:
        result = operation()
    except Exception as exc:
        report["checks"][name] = {"status": "fail", "error_type": type(exc).__name__, "message": str(exc)}
        save_report()
        raise
    report["checks"][name] = {"status": "pass"}
    save_report()
    return result

save_report()
print(json.dumps({k: report[k] for k in ["executable", "prefix", "uid", "site_packages_writable"]}, indent=2))
print("Evidencias:", run_dir)

## 2. Reinstalación desde la rama y reinicio

La siguiente celda imprime el comando que usa el mismo intérprete del kernel.
Con `INSTALL_PACKAGE=True` lo ejecuta y se detiene para exigir reinicio. Confirmar
primero entorno/permisos; no se añade sudo ni se cambia de intérprete automáticamente.
Repetir la reinstalación en 1.8 conservando los recibos de pip y las identidades
antes/después. Esta celda no declara que se haya realizado esa repetición.

Después de instalar: poner `INSTALL_PACKAGE=False`, reiniciar kernel y ejecutar
desde el principio. Los notebooks/evidencias están fuera de site-packages.

In [ ]:
command = [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "--no-deps",
           "pynq-acoustic-camera @ git+https://github.com/SiririComun/sw-acoustic-camera.git@phase/01-baseline"]
print(shlex.join(command))
if INSTALL_PACKAGE:
    assert Path(sys.prefix).resolve() == EXPECTED_ENV.resolve(), "Kernel fuera del entorno PYNQ esperado"
    assert report["site_packages_writable"], "Confirmar permisos de instalación en 1.8"
    with (run_dir / "pip-install.log").open("x") as log:
        subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)
    raise RuntimeError("Instalación terminada: poner INSTALL_PACKAGE=False y reiniciar el kernel")

## 3. Procedencia del paquete instalado y configuración

La importación debe proceder del paquete instalado, no del checkout. La identidad
SW se toma de `direct_url.json`, sin guardar URLs que puedan contener credenciales.
El commit instalado debe coincidir con `EXPECTED_SW_COMMIT`.

In [ ]:
def installed_identity():
    assert Path(sys.prefix).resolve() == EXPECTED_ENV.resolve(), "Seleccionar el kernel PYNQ esperado"
    assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SW_COMMIT), "Completar EXPECTED_SW_COMMIT"
    import pynq_acoustic_camera as camera
    dist = metadata.distribution("pynq-acoustic-camera")
    assert Path(camera.__file__).resolve() == Path(dist.locate_file("pynq_acoustic_camera/__init__.py")).resolve()
    direct = json.loads(dist.read_text("direct_url.json") or "{}")
    installed_commit = direct.get("vcs_info", {}).get("commit_id")
    assert installed_commit == EXPECTED_SW_COMMIT, "Commit instalado distinto del esperado"
    report["software"] = {"version": dist.version, "commit": installed_commit,
                          "module_path": camera.__file__, "repository": "SiririComun/sw-acoustic-camera"}
    return camera

camera = check("installed_identity", installed_identity)
HardwareLoader = camera.HardwareLoader

def pinned_configuration():
    config = HardwareLoader.get_hardware_config()
    assert config["release_tag"] == EXPECTED_HW_TAG
    assert config["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
    assert config["expected_interface_id"] == "legacy-xadc-reference/1"
    report["hardware_config"] = config
    return config

config = check("pinned_configuration", pinned_configuration)
print(json.dumps(report["software"], indent=2))

## 4. Descarga, hashes y reutilización de caché

Usa una caché propia de esta ejecución dentro de sus evidencias. Descargar y
verificar no programa la FPGA. Se guarda el manifiesto autenticado y el inventario
HWH; la procedencia y los hashes identifican el hardware legado, sin inventar BUILD_ID.

In [ ]:
cache_base = run_dir / "overlay-cache"

def acquire_pair():
    bit = HardwareLoader.get_overlay_path(download_dir=cache_base)
    manifest = HardwareLoader.get_manifest(download_dir=cache_base)
    for item in manifest["files"]:
        path = bit.parent / item["name"]
        assert path.stat().st_size == item["size_bytes"]
        assert hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"]
    manifest_path = bit.parent / "build-manifest.json"
    assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256
    (run_dir / "build-manifest.json").write_bytes(manifest_path.read_bytes())
    report["hardware"] = {"source_commit": manifest["source_commit"], "release_tag": EXPECTED_HW_TAG,
                           "manifest_sha256": EXPECTED_MANIFEST_SHA256, "files": manifest["files"],
                           "build_run_id": manifest["build_run_id"], "bit_path": str(bit)}
    return bit

bit_path = check("download_and_hashes", acquire_pair)

def reuse_pair():
    from unittest.mock import patch
    import pynq_acoustic_camera.loader as loader_module
    with patch.object(loader_module, "_download", side_effect=AssertionError("No debe descargar al reutilizar")):
        assert HardwareLoader.get_overlay_path(download_dir=cache_base) == bit_path
check("cache_reuse_without_network", reuse_pair)

import xml.etree.ElementTree as ET
root = ET.parse(bit_path.with_suffix(".hwh")).getroot()
inventory = {m.get("INSTANCE"): {"type": m.get("MODTYPE"), "vlnv": m.get("VLNV")}
             for m in root.findall(".//MODULE")}
assert all(inventory.get(name, {}).get("type") == "axi_dma" for name in ["axi_dma_0", "axi_dma_1", "axi_dma_2"])
(run_dir / "hwh-ip-map.json").write_text(json.dumps(inventory, indent=2) + "\n")
save_report()
print("Pareja verificada:", bit_path)

## 5. Rechazo de archivos corruptos y configuración ausente

Opera sobre una copia temporal. No modifica el único conjunto válido, no cambia
la configuración instalada y no llama a Overlay con datos alterados.

In [ ]:
def rejection_checks():
    import shutil
    from unittest.mock import patch
    import pynq_acoustic_camera.loader as loader_module
    with tempfile.TemporaryDirectory(dir=run_dir) as temporary:
        base = Path(temporary) / "negative-cache"
        entry = HardwareLoader._cache_path(config, base)  # disposición para fixture de prueba
        shutil.copytree(bit_path.parent, entry)
        corrupted = entry / config["files"]["bit"]
        corrupted.write_bytes(b"deliberately invalid test copy")
        try:
            HardwareLoader.get_overlay_path(download_dir=base)
        except camera.HardwareIntegrityError:
            pass
        else:
            raise AssertionError("No se rechazó el BIT corrupto")
        with patch.object(loader_module.resources, "files", return_value=Path(temporary) / "missing-config"):
            try:
                HardwareLoader.get_hardware_config()
            except camera.HardwareConfigError:
                pass
            else:
                raise AssertionError("No se rechazó la configuración ausente")
check("corrupt_copy_and_missing_config_rejected", rejection_checks)

## 6. Carga explícita del overlay — solo en 1.8

Comprobar el inventario y los resultados anteriores. Para programar, cambiar
`PROGRAM_FPGA=True` en los parámetros y ejecutar de nuevo esta celda (o el notebook
desde el inicio). Esto reemplaza el overlay cargado. Se usa Overlay genérico:
no se construye MicrophoneArrayOverlay ni se arma adquisición XADC/PDM.

In [ ]:
if PROGRAM_FPGA:
    assert all(result["status"] == "pass" for result in report["checks"].values())
    def program_reference():
        overlay = HardwareLoader.load_overlay(board="pynq_z2", download_dir=cache_base)
        assert overlay.is_loaded(), "PYNQ no confirma overlay cargado"
        expected = {"axi_dma_0", "axi_dma_1", "axi_dma_2"}
        assert expected <= set(overlay.ip_dict), "Mapa IP cargado incompatible"
        report["overlay_programmed"] = True
        report["loaded_ip_map"] = {name: {key: value for key, value in item.items()
                                        if key in {"type", "phys_addr", "addr_range"}}
                                    for name, item in overlay.ip_dict.items()}
        return overlay
    overlay = check("generic_overlay_load", program_reference)
else:
    report["checks"]["generic_overlay_load"] = {"status": "not_run", "reason": "PROGRAM_FPGA=False"}
    save_report()
    print("Sin programación: carga pendiente para 1.8")

## 7. Guardar evidencias

Guardar una copia de este notebook ejecutado en `run_dir`. El informe no acepta
automáticamente la fase: en 1.8 se adjuntan los recibos de reinstalación repetida,
se revisan los permisos y se comprueba carga desde kernel limpio. En 1.9 se revisa
la combinación exacta HW/SW. No se evalúa calidad acústica con entradas flotantes.

In [ ]:
report["finished_at_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
save_report()
print(json.dumps(report["checks"], indent=2))
print("Informe:", run_dir / "acceptance-report.json")
print("Guardar también una copia ejecutada de este notebook en esa carpeta.")
print("Aceptación de fase 1 pendiente de revisión 1.9.")